In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
import requests
from time import sleep
import os
from urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)
from bidi.algorithm import get_display
from arabic_reshaper import reshape
from googletrans import Translator




    

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName='SD CBOS'
print("Running SD CBOS Web Scraping Tool v.1.0")

now=datetime.datetime.now()
filename= 'SD CBOS SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

Running SD CBOS Web Scraping Tool v.1.0


In [3]:

#------------------------------------------------ Begin_Variable ----------------------------------------
regdict={
         'SD CBOS 1': 'https://cbos.gov.sd/ar/content/%D8%A7%D9%84%D8%A8%D9%86%D9%88%D9%83-%D8%A7%D9%84%D8%B9%D8%A7%D9%85%D9%84%D8%A9-%D8%A8%D8%A7%D9%84%D8%B3%D9%88%D8%AF%D8%A7%D9%86',
         'SD CBOS 2': 'https://cbos.gov.sd/en/content/authorized-exchange-bureaus',
         'SD CBOS 3': 'https://cbos.gov.sd/ar/content/%D8%A7%D9%84%D9%85%D8%A4%D8%B3%D8%B3%D8%A7%D8%AA-%D8%A7%D9%84%D9%85%D8%A7%D9%84%D9%8A%D8%A9'
         }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}


Typology ={

            'SD CBOS 1': 'Banks',
            'SD CBOS 2': 'Financial Institutions',
            'SD CBOS 3': "Foreign Banks' Representation Offices in Lebanon",

            
}

processdate=now.strftime('%Y-%m-%d')

headers = {
    "authority": "cbos.gov.sd",
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9,zh;q=0.8,zh-CN;q=0.7",
    "cache-control": "max-age=0",
    "if-none-match": 'W/"1761117078-0"',
    "priority": "u=0, i",
    "sec-ch-ua": '"Microsoft Edge";v="141", "Not?A_Brand";v="8", "Chromium";v="141"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "cross-site",
    "sec-fetch-user": "?1",
    "upgrade-insecure-requests": "1",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36 Edg/141.0.0.0",
}

cookies = {
    "_ga": "GA1.3.1067748827.1761040053",
    "_gid": "GA1.3.861544352.1761040053",
    "has_js": "1",
}


In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict



# Initialize the translator
translator = Translator()
def translate_text(text):
    return translator.translate(text, src='ar', dest='en').text


In [ ]:


#------------------------------------------------ Begin_Main ----------------------------------------
session = requests.Session()
for reg in regdict:
    print('Working with {}'.format(reg))
    

    response = requests.get(regdict[reg],headers=headers, cookies=cookies, verify=False, timeout=30)

    if response.status_code != 200:

        print(f"{reg}: got {response.status_code}, skipping…")

        continue  
    else:
        print(response.status_code)

    data = response.text  
    soup = BeautifulSoup(data, "html.parser")
    if reg == 'SD CBOS 1':
        div=soup.find("div",{"class":"main-content-area-wrapper"})
        #print(div.text)
        ols=div.find("ol")
        lis=ols.find_all("li")
        #print(len(lis))
        for li in lis:
            ori_name = li.text.strip()
            reshaped_text = reshape(ori_name)

            trans_name = translate_text(reshaped_text)


            sqldict['Name'].append(reshaped_text)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Cntry'].append('SD')
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
            sqldict['Name - Mother Company'].append(trans_name)

            sqldict = bourange_same_length_array(sqldict)

    elif reg == 'SD CBOS 2':
        div=soup.find("div",{"class":"main-content-area-wrapper"})
        #print(div.text)
        ols=div.find("ol")
        lis=ols.find_all("li")
        
        for li in lis:
            #print(li.text.strip())
            sqldict['Name'].append(li.text.strip())
            sqldict['ListProcessDate'].append(processdate)

            sqldict['Cntry'].append('SD')
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])

            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'SD CBOS 3':
        tables = soup.find_all('table')
        tbody = tables[0].find('tbody') or tables[0]
        rows = tbody.find_all('tr', recursive=False)
        for tr in rows:
            if tr.find('table'):
                continue          # skip nested tables but keep scanning later rows

            tds = tr.find_all('td', recursive=False)
            if not tds:
                continue
            if any(cell.find('strong') for cell in tds):
                continue          # header lines skiping

            name_raw = tds[0].get_text(strip=True)
            try:
                name = translate_text(name_raw)
            except Exception:
                continue
            if not name or len(name) < 7 or name.lower() == "company":
                continue
            # print("Name:", name)
            sqldict['Name'].append(name_raw)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Name - Mother Company'].append(name)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])
            

            if len(tds) > 1:
                addr_raw = tds[1].get_text(strip=True)
                try:
                    addr = translate_text(addr_raw)
                except Exception:
                    addr = ""
                if addr and len(addr) >= 7 and addr.lower() != "the address":
                    sqldict['Address_1'].append(addr_raw)
                    sqldict['Address_1 - Mother company'].append(addr)
                else:
                    sqldict['Address_1'].append('')
                    sqldict['Address_1 - Mother company'].append('')

            if len(tds) > 2:
                contact_raw = tds[2].get_text(strip=True)
                try:
                    contact = translate_text(contact_raw)
                except Exception:
                    contact = ""
                if contact and len(contact) >= 7 and contact.lower() != "contact":
                    #print("Contact:", contact)
                    sqldict['Phone'].append(contact)
                else:
                    sqldict['Phone'].append('')
            sqldict = bourange_same_length_array(sqldict)
                

        index = [-1,-2]
        for i in index:
            trs = tables[i].find_all('tr')
            for tr in trs:
                tds = tr.find_all('td')
                if tds:
                    name_raw = tds[1].text
                    name = translate_text(name_raw)
                    if name != 'total' and 'Names of' not in name:
                        #print(name,name_raw)
                        sqldict['Name'].append(name_raw)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Name - Mother Company'].append(name)
                        sqldict['Cntry'].append('SD')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split(' ')[0]) 
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict['ListName'].append(Typology[reg])
                        sqldict = bourange_same_length_array(sqldict)
                        continue    


Working with SD CBOS 1
200
38
Working with SD CBOS 2
200
Working with SD CBOS 3
200


In [6]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 99 values.
Key 'priority' has 99 values.
Key 'ListLabel' has 99 values.
Key 'Typology' has 99 values.
Key 'EntryType' has 99 values.
Key 'Name' has 99 values.
Key 'InternalID_1' has 99 values.
Key 'InternalID_1_type' has 99 values.
Key 'InternalID_2' has 99 values.
Key 'InternalID_2_type' has 99 values.
Key 'InternalID_3' has 99 values.
Key 'InternalID_3_type' has 99 values.
Key 'CoType' has 99 values.
Key 'License_Type' has 99 values.
Key 'Address_1' has 99 values.
Key 'Address_2' has 99 values.
Key 'City' has 99 values.
Key 'Zip' has 99 values.
Key 'Cntry' has 99 values.
Key 'Phone' has 99 values.
Key 'Fax' has 99 values.
Key 'Website' has 99 values.
Key 'Email' has 99 values.
Key 'RegulationType' has 99 values.
Key 'RegulationTypeCode' has 99 values.
Key 'RegulationDate' has 99 values.
Key 'CancellationDate' has 99 values.
Key 'RegCtry' has 99 values.
Key 'RegCode' has 99 values.
Key 'ListCode' has 99 values.
Key 'ListLanguage' has 99 values.
Key 'ListValidityDate' h

In [8]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)


C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_29484\244866655.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [10]:
for col in ["Name", "Name - Mother Company"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"^\s*\d+[.\s]*", "", regex=True)
        .str.strip()
    )


In [ ]:
df.to_excel('total_clean.xlsx')

: 